# DeepLearning: the AI/PyTorch bridge for molecular ML

Role of this notebook: I am your graduate tutor in data science / computer science. You already program Python well and have beginner PyTorch/ML/transformer experience. The goal is not to make you memorize all of deep learning; it is to give you the exact conceptual and practical kit needed for the project notebooks:

- `MPNNs`: supervised graph neural networks for quantum chemistry.
- `DeepAffinity`: sequence + compound encoders for compound--protein affinity.
- `GraphVAE`, `JunctionTreeVariationalAutoencoder`, `MolGAN`: molecular generative models.
- `DeepDDS`, `HiGNN`: graph models, attention, drug synergy/property prediction.

Recommended external resources:

- [Dive into Deep Learning](https://d2l.ai/): open textbook with runnable PyTorch code. Most relevant: chapters 2--6, 10--13, 20.
- [Understanding Deep Learning](https://udlbook.github.io/udlbook/): modern theory-first text. Use chapters 2--7 for foundations, 12 for transformers, 17--20 for generative models/graphs if present in your edition. Page numbers vary between HTML/PDF builds, so the chapter anchors are more reliable than printed pages.
- [fast.ai Practical Deep Learning for Coders](https://course.fast.ai/): excellent top-down practice, especially lessons 1--5.
- [Stanford CS224W: Machine Learning with Graphs](https://web.stanford.edu/class/cs224w/): graph ML/GNN lectures; especially message passing, GCN/GraphSAGE/GAT, and graph generation.
- [NYU Deep Learning](https://atcold.github.io/NYU-DLSP20/): great lectures on optimization, CNNs, sequence models, and representation learning.

Notebook style: each lesson has a mental model, formulas, a runnable mini-demo, and a connection to the molecular notebooks.

In [ ]:
from __future__ import annotations

import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

try:
    import torch_geometric
    from torch_geometric.data import Data, Batch
    from torch_geometric.nn import GCNConv, GINEConv, global_mean_pool
    HAS_PYG = True
except Exception as exc:
    HAS_PYG = False
    PYG_IMPORT_ERROR = repr(exc)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Learning" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "Learning"
MODEL_DIR = PROJECT_ROOT / "models" / "Learning"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
np.random.seed(7)
random.seed(7)

print("PyTorch:", torch.__version__)
print("Device:", device)
print("PyG available:", HAS_PYG)

## Lesson 1 — Tensors, shapes, and the habit of explicit dimensions

Deep learning code is mostly tensor algebra plus bookkeeping. The most important debugging skill is asking:

1. What does each axis mean?
2. Which operation mixes which axis?
3. Does batching change the meaning?

Typical molecular ML tensor meanings:

| object | common shape | meaning |
|---|---:|---|
| atom features | `[num_atoms, atom_dim]` | one row per atom |
| bond index | `[2, num_edges]` | source/target atom indices |
| protein tokens | `[batch, length]` | amino-acid token IDs |
| graph embedding | `[batch, hidden]` | one vector per molecule |
| pair embedding | `[batch, hidden]` | drug pair / compound-protein representation |

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
print("x shape:", x.shape)
print("batch 0:\n", x[0])
print("mean over sequence axis ->", x.float().mean(dim=1).shape)
print("mean over feature axis  ->", x.float().mean(dim=2).shape)

## Lesson 2 — Supervised learning as empirical risk minimization

For data points $(x_i, y_i)$, a model $f_\theta$, and loss $\ell$, training minimizes

$$
\hat{R}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i), y_i).
$$

Regression notebooks use MSE/RMSE/MAE. Classification notebooks use cross-entropy/BCE and AUC. Generative notebooks optimize likelihood surrogates, ELBOs, or adversarial losses.

Resources:

- D2L chapters 3--4: linear regression and softmax regression.
- UDL chapters 2--5: supervised learning, losses, gradients, and training.

In [ ]:
# A tiny regression problem: learn y = 2x - 1 + noise.
n = 256
X = torch.linspace(-3, 3, n).unsqueeze(1)
y = 2 * X - 1 + 0.25 * torch.randn_like(X)

model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
X_d, y_d = X.to(device), y.to(device)

losses = []
for step in range(400):
    pred = model(X_d)
    loss = F.mse_loss(pred, y_d)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach().cpu()))

plt.plot(losses)
plt.yscale("log")
plt.title("Training loss")
plt.xlabel("step")
plt.ylabel("MSE")
plt.show()

with torch.no_grad():
    plt.scatter(X.numpy(), y.numpy(), s=8, alpha=0.35, label="data")
    plt.plot(X.numpy(), model(X_d).cpu().numpy(), color="black", label="MLP")
    plt.legend()
    plt.show()

## Lesson 3 — Autograd and backpropagation

PyTorch records operations on tensors with `requires_grad=True`. Calling `loss.backward()` computes gradients by reverse-mode automatic differentiation.

The chain rule is the whole spell:

$$
\frac{\partial L}{\partial \theta}
=
\frac{\partial L}{\partial h}
\frac{\partial h}{\partial \theta}.
$$

Practical rules:

- Use `model.train()` while training, `model.eval()` while validating.
- Call `optimizer.zero_grad(set_to_none=True)` before backprop.
- Do not use `.item()` inside a differentiable expression.
- Use `with torch.no_grad()` for evaluation.

In [ ]:
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
pred = w * X[:5] + b
loss = ((pred - y[:5]) ** 2).mean()
loss.backward()
print("loss:", float(loss))
print("dL/dw:", w.grad.item(), "dL/db:", b.grad.item())

## Lesson 4 — Optimization, regularization, and validation

The project notebooks repeatedly use:

- Adam/AdamW: adaptive optimizer, strong default.
- weight decay: penalizes large weights.
- dropout: stochastic feature removal.
- early stopping: stop when validation performance stops improving.
- scaffold splits: in chemistry, more realistic than random splits because similar molecules leak less easily.

For optimizer theory:

- D2L chapter 12: SGD, momentum, RMSProp, Adam, learning-rate schedules.
- NYU DL lectures on optimization.

In [ ]:
def train_val_split(n, frac=0.8):
    idx = torch.randperm(n)
    cut = int(frac * n)
    return idx[:cut], idx[cut:]

train_idx, val_idx = train_val_split(len(X))
model = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Dropout(0.05), nn.Linear(64, 1)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

history = {"train": [], "val": []}
best_val = float("inf")
best_state = None

for epoch in range(300):
    model.train()
    pred = model(X_d[train_idx])
    loss = F.mse_loss(pred, y_d[train_idx])
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        val = F.mse_loss(model(X_d[val_idx]), y_d[val_idx])
    history["train"].append(float(loss.cpu()))
    history["val"].append(float(val.cpu()))
    if history["val"][-1] < best_val:
        best_val = history["val"][-1]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

torch.save({"model": best_state, "best_val_mse": best_val}, MODEL_DIR / "tiny_regression.pt")
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="val")
plt.yscale("log")
plt.legend()
plt.title("Train/validation curves")
plt.show()
print("Saved:", MODEL_DIR / "tiny_regression.pt")

## Lesson 5 — Embeddings: turning discrete symbols into vectors

Molecular notebooks often embed:

- atom type, degree, formal charge, chirality;
- bond type, conjugation, ring membership;
- amino-acid tokens for protein sequences;
- fragment/clique labels in junction-tree models.

An embedding table is just a learned lookup matrix $E \in \mathbb{R}^{V \times d}$.

In [ ]:
aa = "ACDEFGHIKLMNPQRSTVWY"
stoi = {ch: i + 1 for i, ch in enumerate(aa)}  # 0 reserved for padding
seqs = ["MKWVTFISLL", "ACDEFGHIK", "GGGGGG"]
max_len = max(map(len, seqs))
tokens = torch.zeros((len(seqs), max_len), dtype=torch.long)
for i, seq in enumerate(seqs):
    tokens[i, :len(seq)] = torch.tensor([stoi.get(ch, 0) for ch in seq])

emb = nn.Embedding(num_embeddings=len(stoi) + 1, embedding_dim=8, padding_idx=0)
z = emb(tokens)
mask = (tokens != 0).unsqueeze(-1)
pooled = (z * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
print("tokens:", tokens.shape)
print("embedded:", z.shape)
print("mean pooled sequence embedding:", pooled.shape)

## Lesson 6 — Attention and transformers, but only what you need here

Attention computes a weighted mixture of value vectors:

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.
$$

In this repo:

- DeepAffinity uses sequence/compound representations and interpretable affinity signals.
- DeepDDS and HiGNN use attention-like mechanisms for drug-pair or feature-wise importance.
- Transformers are conceptually useful even when the model is a GNN: both pass contextual information between tokens/nodes.

Resources:

- D2L chapter 11: attention and transformers.
- UDL transformer chapter.
- Stanford CS224N transformer lectures if you want NLP depth.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = scores.softmax(dim=-1)
    return weights @ V, weights

torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out, weights = scaled_dot_product_attention(Q, K, V)
plt.imshow(weights[0].detach().numpy(), cmap="viridis")
plt.colorbar()
plt.title("Toy attention matrix")
plt.xlabel("key position")
plt.ylabel("query position")
plt.show()

## Lesson 7 — Graph neural networks and message passing

Most molecular graph models follow:

$$
m_v^{(t+1)} = \sum_{u \in \mathcal{N}(v)} M_t(h_v^{(t)}, h_u^{(t)}, e_{uv}),
$$

$$
h_v^{(t+1)} = U_t(h_v^{(t)}, m_v^{(t+1)}).
$$

Readout makes a graph-level vector:

$$
h_G = R(\{h_v^{(T)} : v \in G\}).
$$

Connections:

- MPNN paper: this framework was named and standardized for quantum chemistry.
- DeepDDS/HiGNN: GNNs plus attention and hierarchical structure.
- GraphVAE/MolGAN/JT-VAE: generate graphs rather than just predict labels.

Resources:

- Stanford CS224W graph neural network lectures.
- D2L has graph and recommender-adjacent material; CS224W is the better primary graph ML resource.

In [ ]:
if not HAS_PYG:
    print("PyTorch Geometric not available:", PYG_IMPORT_ERROR)
else:
    # Two tiny molecular-ish graphs, with categorical-ish node features replaced by random vectors.
    g1 = Data(
        x=torch.randn(3, 8),
        edge_index=torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long),
        y=torch.tensor([0.5]),
    )
    g2 = Data(
        x=torch.randn(4, 8),
        edge_index=torch.tensor([[0, 1, 2, 2, 3, 0], [1, 0, 1, 3, 2, 3]], dtype=torch.long),
        y=torch.tensor([1.0]),
    )
    batch = Batch.from_data_list([g1, g2]).to(device)
    conv = GCNConv(8, 16).to(device)
    head = nn.Linear(16, 1).to(device)
    h = conv(batch.x, batch.edge_index).relu()
    graph_h = global_mean_pool(h, batch.batch)
    pred = head(graph_h).view(-1)
    loss = F.mse_loss(pred, batch.y.float())
    loss.backward()
    print("node embeddings:", h.shape)
    print("graph embeddings:", graph_h.shape)
    print("loss:", float(loss.detach().cpu()))

## Lesson 8 — Variational autoencoders: the generative-model skeleton

VAEs learn a latent distribution. Encoder:

$$
q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \mathrm{diag}(\sigma_\phi^2(x))).
$$

Decoder:

$$
p_\theta(x|z).
$$

Objective:

$$
\mathcal{L}_{ELBO}
=
\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]
-
D_{KL}(q_\phi(z|x) \| p(z)).
$$

Connections:

- GraphVAE: decodes adjacency/node/edge tensors.
- JT-VAE: decodes a chemically constrained junction tree plus molecular graph.

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=2, hidden=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU())
        self.mu = nn.Linear(hidden, latent_dim)
        self.logvar = nn.Linear(hidden, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, input_dim))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        std = (0.5 * logvar).exp()
        z = mu + std * torch.randn_like(std)
        recon = self.dec(z)
        return recon, mu, logvar

theta = torch.linspace(0, 2 * math.pi, 512)
circle = torch.stack([torch.cos(theta), torch.sin(theta)], dim=1) + 0.05 * torch.randn(512, 2)
vae = TinyVAE().to(device)
opt = torch.optim.AdamW(vae.parameters(), lr=1e-3)
x = circle.to(device)
losses = []
for step in range(700):
    recon, mu, logvar = vae(x)
    recon_loss = F.mse_loss(recon, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon_loss + 0.05 * kl
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach().cpu()))

with torch.no_grad():
    z = torch.randn(512, 2, device=device)
    samples = vae.dec(z).cpu()
plt.scatter(circle[:, 0], circle[:, 1], s=5, alpha=0.4, label="data")
plt.scatter(samples[:, 0], samples[:, 1], s=5, alpha=0.4, label="VAE samples")
plt.axis("equal")
plt.legend()
plt.title("Tiny VAE intuition demo")
plt.show()

## Lesson 9 — GANs and reinforcement signals

MolGAN is an implicit generative model:

- generator creates graph tensors;
- discriminator judges realism;
- reward network or RL term encourages molecular desirability.

Vanilla GAN objective:

$$
\min_G \max_D \mathbb{E}_{x\sim p_{data}}\log D(x)
+ \mathbb{E}_{z\sim p(z)}\log(1-D(G(z))).
$$

Practical moral: GAN training is less forgiving than supervised training. You monitor validity, uniqueness, novelty, mode collapse, and chemical constraints, not only loss curves.

## Lesson 10 — How to read the project notebooks like a researcher

For each paper notebook, ask:

1. What is the input representation? SMILES, molecular graph, protein sequence, cell-line feature?
2. What inductive bias is built into the architecture? Message passing, hierarchy, junction tree, attention, adversarial learning?
3. What is the loss function?
4. What split/evaluation protocol prevents leakage?
5. What is saved to `data/` and `models/`, and how do I resume?
6. What would fail if the chemistry representation is wrong?

Suggested route through this repo:

1. Run `Environment_Hardware_Check.ipynb`.
2. Study this notebook through Lesson 7.
3. Run `Cheminformatics` Lessons 1--5.
4. Run `MPNNs/start_MPNN.ipynb`.
5. Then do `DeepDDS`/`HiGNN`.
6. Finally do the generative notebooks: `GraphVAE`, `JT-VAE`, `MolGAN`.

## Lesson 11 — Dataset design, leakage, and why molecular ML is unusually sneaky

In image classification, a random train/test split is often defensible. In molecular ML, it can be dangerously optimistic because close analogs may appear in both train and test. A model can then learn a local interpolation rule rather than chemistry that transfers to new scaffolds.

Important split types:

| split | what it tests | typical use |
|---|---|---|
| random | interpolation among similar molecules | quick debugging |
| scaffold | generalization to new cores | molecular property prediction |
| time split | future compounds from past compounds | medicinal chemistry programs |
| protein-family split | new protein classes | affinity/generalization |
| cell-line split | new biological contexts | drug response/synergy |

Leakage examples in this repo's domain:

- same compound appears with slightly different salts/protonation;
- stereochemistry stripped from one copy but not another;
- assay replicates split across train/test;
- protein sequence near-duplicates split across train/test;
- generated molecule novelty measured against canonical SMILES instead of standardized molecules.

Research habit: before tuning the model, plot distributions and nearest-neighbor similarities between splits.

In [ ]:
# A tiny visual intuition for random vs. grouped splitting.
rng = np.random.default_rng(42)
centers = np.array([[-2, -1], [0, 1], [2, -1]])
Xtoy, groups = [], []
for gid, c in enumerate(centers):
    pts = c + 0.35 * rng.normal(size=(40, 2))
    Xtoy.append(pts)
    groups += [gid] * len(pts)
Xtoy = np.vstack(Xtoy)
groups = np.array(groups)

random_test = rng.choice(len(Xtoy), size=24, replace=False)
group_test = np.where(groups == 2)[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, test_idx, title in [(axes[0], random_test, "random split"), (axes[1], group_test, "scaffold-like group split")]:
    train_mask = np.ones(len(Xtoy), dtype=bool)
    train_mask[test_idx] = False
    ax.scatter(Xtoy[train_mask, 0], Xtoy[train_mask, 1], s=20, label="train", alpha=0.6)
    ax.scatter(Xtoy[test_idx, 0], Xtoy[test_idx, 1], s=35, label="test", alpha=0.9)
    ax.set_title(title)
    ax.axis("equal")
    ax.legend()
plt.suptitle("Generalization is defined by the split, not the optimizer")
plt.tight_layout()
plt.show()

## Lesson 12 — Metrics you will see in the project notebooks

Regression:

$$
\mathrm{MAE}=\frac{1}{N}\sum_i |\hat{y}_i-y_i|,\quad
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_i(\hat{y}_i-y_i)^2}
$$

RMSE punishes outliers more strongly. MAE is often easier to interpret chemically if the units are meaningful.

Binary classification:

- accuracy: easy but bad for imbalanced data;
- precision/recall: useful for active-hit discovery;
- ROC-AUC: probability a random positive ranks above a random negative;
- PR-AUC: better under severe class imbalance.

Generative models:

- validity, uniqueness, novelty;
- reconstruction accuracy for VAEs;
- distribution matching for descriptors;
- property optimization;
- diversity and scaffold diversity.

For molecular science, always pair metrics with examples. A beautiful ROC-AUC plus chemically absurd false positives is not a win; it is a warning siren with a violin section.

In [ ]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {"MAE": mae, "RMSE": rmse}

def roc_auc_rank(y_true, score):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    pos = score[y_true]
    neg = score[~y_true]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    wins = 0.0
    for p in pos:
        wins += np.sum(p > neg) + 0.5 * np.sum(p == neg)
    return wins / (len(pos) * len(neg))

y_true = np.array([0.0, 1.0, 2.0, 4.0])
print(regression_metrics(y_true, [0.1, 1.2, 1.8, 7.0]))
print("ROC-AUC:", roc_auc_rank([0, 1, 0, 1, 1], [0.1, 0.9, 0.4, 0.6, 0.7]))

## Lesson 13 — Batching variable-size objects

Molecules, proteins, and drug pairs rarely have the same size:

- graphs have different numbers of atoms/bonds;
- proteins have different sequence lengths;
- generated graphs may have padded maximum sizes;
- junction trees have different numbers of cliques.

Two standard strategies:

1. **Padding + mask** for sequences or fixed-size graph tensors.
2. **Concatenation + index vectors** for PyG graph batches.

Masking matters. If you mean-pool padded tokens without a mask, the model learns that sequence length and padding are chemistry. That is a very small goblin, but it can wreck a result.

In [ ]:
# Sequence padding with a mask.
lengths = torch.tensor([5, 9, 3])
max_len = int(lengths.max())
emb_dim = 4
seq = torch.zeros(len(lengths), max_len, emb_dim)
for i, L in enumerate(lengths):
    seq[i, :L] = torch.randn(L, emb_dim)
mask = torch.arange(max_len)[None, :] < lengths[:, None]
wrong_pool = seq.mean(dim=1)
right_pool = (seq * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(-1)
print("wrong pooled shape:", wrong_pool.shape)
print("right pooled shape:", right_pool.shape)
print("difference for padded examples:", (wrong_pool - right_pool).norm(dim=1))

## Lesson 14 — A reusable PyTorch training loop pattern

Almost every supervised notebook can be reduced to:

1. create dataset and split;
2. make dataloaders;
3. initialize model;
4. train one epoch;
5. evaluate;
6. save best checkpoint;
7. reload checkpoint for test/prediction.

Cluster-specific additions:

- mixed precision with `torch.amp.autocast`;
- `num_workers` and `pin_memory` for GPU data loading;
- checkpoint every epoch or every validation improvement;
- log the git hash/environment report if possible.

The point of a training loop is not just to optimize. It is to make experiments restartable.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def save_checkpoint(path, model, optimizer=None, **metadata):
    payload = {"model_state": model.state_dict(), "metadata": metadata}
    if optimizer is not None:
        payload["optimizer_state"] = optimizer.state_dict()
    torch.save(payload, path)

def load_checkpoint(path, model, map_location="cpu"):
    payload = torch.load(path, map_location=map_location)
    model.load_state_dict(payload["model_state"])
    return payload.get("metadata", {})

demo = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 1))
print("parameters:", count_parameters(demo))
save_checkpoint(MODEL_DIR / "training_loop_demo.pt", demo, note="minimal checkpoint pattern")
print(load_checkpoint(MODEL_DIR / "training_loop_demo.pt", demo))

## Lesson 15 — Mixed precision and GPU performance

On modern NVIDIA GPUs, float16/bfloat16 matrix operations can be much faster than float32. PyTorch's automatic mixed precision keeps many operations in lower precision while preserving stability for sensitive operations.

Typical pattern:

```python
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
    loss = model(batch)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

Use mixed precision for large GNNs/sequence models on the cluster. Be cautious with:

- tiny losses and unstable VAEs/GANs;
- custom operations;
- metrics computed in low precision;
- CPU execution, where mixed precision usually does not help.

In [ ]:
if torch.cuda.is_available():
    a = torch.randn(4096, 4096, device="cuda")
    b = torch.randn(4096, 4096, device="cuda")
    torch.cuda.synchronize()
    t0 = time.perf_counter() if "time" in globals() else None
    with torch.amp.autocast("cuda"):
        c = a @ b
    torch.cuda.synchronize()
    print("autocast matmul dtype:", c.dtype)
else:
    print("CUDA not available here; run this cell on the cluster to see mixed-precision behavior.")

## Lesson 16 — Oversmoothing and over-squashing in GNNs

Two common GNN failure modes:

**Oversmoothing:** after many message-passing layers, node embeddings become too similar.

**Over-squashing:** long-range information from many nodes is compressed through narrow graph bottlenecks.

Symptoms:

- deeper GNN performs worse;
- node embeddings have low variance;
- model cannot capture long-range functional-group interactions.

Mitigations:

- residual/skip connections;
- normalization;
- virtual nodes/global tokens;
- hierarchical pooling/fragments;
- attention or edge-aware message functions;
- avoid excessive depth unless there is a reason.

This is one reason HiGNN's fragment hierarchy is appealing: it gives information a coarser route through the molecule.

In [ ]:
if HAS_PYG:
    # Oversmoothing toy: repeatedly apply normalized adjacency-like averaging on a path graph.
    n = 25
    h = torch.eye(n)
    A = torch.zeros(n, n)
    for i in range(n - 1):
        A[i, i + 1] = A[i + 1, i] = 1
    A += torch.eye(n)
    D_inv = torch.diag(1 / A.sum(dim=1))
    P = D_inv @ A
    variances = []
    for _ in range(40):
        h = P @ h
        variances.append(float(h.var(dim=0).mean()))
    plt.plot(variances)
    plt.yscale("log")
    plt.xlabel("message passing steps")
    plt.ylabel("mean feature variance")
    plt.title("Toy oversmoothing by repeated neighbor averaging")
    plt.show()
else:
    print("PyG not required for the math, but earlier imports failed:", globals().get("PYG_IMPORT_ERROR"))

## Lesson 17 — Multi-task learning and missing labels

Molecular datasets often contain multiple targets:

- toxicity endpoints;
- quantum properties;
- assay panels;
- cell-line/drug-pair responses.

The loss with missing labels usually uses a mask:

$$
L = \frac{\sum_{i,t} m_{i,t}\ell(\hat{y}_{i,t}, y_{i,t})}{\sum_{i,t}m_{i,t}}.
$$

This appears simple but changes everything:

- batches have different effective target counts;
- rare tasks may be undertrained;
- task scaling matters for regression;
- one task can dominate shared representations.

In [ ]:
pred = torch.tensor([[0.2, -1.0, 0.4], [1.2, 0.1, -0.5]], requires_grad=True)
target = torch.tensor([[1.0, 0.0, 1.0], [0.0, -1.0, 1.0]])
mask = torch.tensor([[1, 0, 1], [1, 1, 0]], dtype=torch.float32)
loss_per_entry = F.binary_cross_entropy_with_logits(pred, target, reduction="none")
masked_loss = (loss_per_entry * mask).sum() / mask.sum()
masked_loss.backward()
print("masked BCE:", float(masked_loss))
print("gradient exists where label is present; masked positions contribute zero to loss")

## Lesson 18 — Reading paper architectures as computation graphs

When a paper diagram looks intimidating, rewrite it as:

1. input tensors;
2. encoder(s);
3. interaction/fusion;
4. pooling/readout;
5. objective.

Examples:

- **MPNN:** atom/bond tensors → message passing → graph readout → property loss.
- **DeepAffinity:** compound encoder + protein encoder → interaction/attention → affinity loss.
- **DeepDDS:** drug A graph + drug B graph + cell-line features → fusion/attention → synergy class.
- **HiGNN:** atom graph + fragment graph → feature-wise attention fusion → property prediction.
- **GraphVAE:** graph encoder → latent Gaussian → padded graph decoder → ELBO/reconstruction.
- **JT-VAE:** molecular graph → junction tree + graph latent variables → constrained decode.
- **MolGAN:** noise → graph generator, graph → discriminator/reward → adversarial/RL losses.

If you can name the tensors at every arrow, you understand the method well enough to implement a faithful reproduction.